# Topic Spam Email Classifier
## End-to-End NLP & Supervised Classification Machine Learning Pipeline

This notebook demonstrates the core AI concepts behind modern email spam filters:
1. **Natural Language Processing (NLP)**: Preprocessing, tokenization, stopword filtering, and Porter stemming.
2. **Feature Extraction**: Sublinear TF-IDF vectorization and domain heuristics (URLs, currency signs, caps ratio).
3. **Supervised Classification**: Training and benchmarking Multinomial Naive Bayes, Logistic Regression, LinearSVC, and Random Forest.
4. **Fine-grained Topic Classification**: Classifying spam categories (Phishing, Lottery, Crypto, Marketing, Work, Personal).
5. **Explainable AI (XAI)**: Visualizing per-email token impacts.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessor import TextPreprocessor
from src.features import build_tfidf_vectorizer, DomainFeatureExtractor
from src.model import MODEL_ZOO, SpamClassifierPipeline
from src.classifier import EmailClassifierService

### 1. NLP Preprocessing and Normalization
Raw email text often contains unstructured noise such as URLs, email addresses, non-standard currency symbols, and shouting capital letters.
Our NLP pipeline normalizes these elements into semantic tokens (`token_url`, `token_email`, `token_money`) and applies Porter Stemming.

In [ ]:
preprocessor = TextPreprocessor(use_stemming=True)
sample_email = "URGENT: Click http://verify-paypal.com to claim your $2,500 refund immediately!!!"

print("Raw Email:", sample_email)
print("Cleaned Text:", preprocessor.clean_text(sample_email))
print("Fully Processed:", preprocessor.process(sample_email))

### 2. Feature Vectorization: TF-IDF with N-Grams
TF-IDF balances term frequency with document rarity:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \log\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$$

We also extract unigrams and bigrams (e.g., `"wire transfer"`, `"click link"`) to capture multi-word contextual phrases.

In [ ]:
df = pd.read_csv("../data/emails.csv")
print(f"Loaded dataset with {len(df)} emails.")
print(df.head(3))

### 3. Live Inference with Explanations and Topic Tagging

In [ ]:
service = EmailClassifierService(
    spam_model_path="../models/spam_classifier_model.joblib",
    topic_model_path="../models/topic_classifier_model.joblib"
)

test_text = "URGENT: Your bank account access has been restricted. Visit our secure portal to verify identity."
result = service.classify_email(test_text)

print("Verdict:", result["label"].upper(), f"({result['confidence_percent']}%)")
print("Predicted Topic:", result["topic_display"])
print("Top Influential Tokens:", result["explanation"]["top_spam_words"])